---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

*This notebook is the assignment template. Each group works on its own copy, and cells or sections may be added as needed.*

---

## Group XX

- Name (NIM)
- Name (NIM)
- Name (NIM)
- Name (NIM)
- Name (NIM)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

We predict whether an accommodation booking will be canceled.

## Aim

Prepare consistent training and validation data for the five classifiers.

## Metric

Kaggle ranks submissions by macro F1, the average F1 across both classes.

## Dataset

The training set contains 27,373 bookings and 17 features. The test set has 6,844 bookings without labels. Booking IDs are strings.

## Approach

Inspect data quality, split training and validation, fit preprocessing on the training split, then compare models using the same holdout.

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [ ]:
# %pip install -q numpy pandas

## Import Libraries

Every import of the notebook is kept in this cell.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
if (Path.cwd() / 'data/train.csv').is_file():
    os.chdir('src')


## Settings

Paths and shared constants are kept in this class.

In [ ]:
class Settings:
    SEED       = 42
    DATA_DIR   = Path('../data')
    OUTPUT_DIR = Path('../output')

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={'id': str})
test_df  = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={'id': str})

print(f'Train: {train_df.shape}  Test: {test_df.shape}')

---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [ ]:
import sys
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

sys.path.insert(0, str(Path.cwd()))
from brownbear.preprocessing import (
    BINARY_COLUMNS, CATEGORICAL_COLUMNS, FEATURE_COLUMNS, NUMERIC_COLUMNS,
    make_basic_preprocessor, make_discrete_preprocessor,
    make_numeric_preprocessor, select_features, split_training_data,
)

description = pd.DataFrame({
    'dtype': train_df.dtypes.astype(str),
    'unique': train_df.nunique(dropna=True),
    'missing': train_df.isna().sum(),
})
print(description)
print(train_df[NUMERIC_COLUMNS].describe().T)

## Explore Data

This part covers distributions and relationships between attributes.

In [ ]:
print(train_df['is_canceled'].value_counts().sort_index().rename('count').to_frame())
for column in CATEGORICAL_COLUMNS:
    print(column, train_df[column].value_counts(dropna=False).to_dict())

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
raw_train = select_features(train_df)
raw_test = select_features(test_df)
quality = pd.DataFrame(index=FEATURE_COLUMNS)
quality['train_missing'] = raw_train.isna().sum()
quality['train_missing_pct'] = raw_train.isna().mean().mul(100)
quality['test_missing'] = raw_test.isna().sum()
quality['test_missing_pct'] = raw_test.isna().mean().mul(100)
print(quality.loc[quality[['train_missing', 'test_missing']].sum(axis=1) > 0].round(2))
print('Observed values and arrival dates passed validation.')

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
train_idx, val_idx = split_training_data(train_df, seed=CFG.SEED)
train_set, val_set = train_df.iloc[train_idx].copy(), train_df.iloc[val_idx].copy()
groups = raw_train.groupby(FEATURE_COLUMNS, dropna=False, sort=False).ngroup()
assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[val_idx])
assert sorted(np.r_[train_idx, val_idx]) == list(range(len(train_df)))
assert set(train_set['is_canceled']) == set(val_set['is_canceled']) == {0, 1}
split_summary = pd.DataFrame({
    name: {'rows': len(frame), 'canceled': int(frame['is_canceled'].sum()),
           'canceled_pct': frame['is_canceled'].mean() * 100}
    for name, frame in [('training', train_set), ('validation', val_set)]
}).T
print(split_summary.round(3))

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [ ]:
X_train_raw, X_val_raw = select_features(train_set), select_features(val_set)
y_train = train_set['is_canceled'].astype(int)
y_val = val_set['is_canceled'].astype(int)
test_ids = test_df['id'].copy()
assert not X_train_raw.isna().all().any(), 'Review features that are entirely missing before fitting.'
print('Using all 17 features; ID and target are kept separately.')

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

In [ ]:
prep = make_basic_preprocessor()
X_train = prep.fit_transform(X_train_raw)
X_val = prep.transform(X_val_raw)
X_test = prep.transform(raw_test)
imputer = prep.named_steps['impute']
fills = pd.DataFrame({'feature': NUMERIC_COLUMNS + BINARY_COLUMNS,
                     'strategy': ['median'] * len(NUMERIC_COLUMNS) + ['mode'] * len(BINARY_COLUMNS),
                     'value': np.r_[imputer.named_transformers_['numeric'].statistics_,
                                    imputer.named_transformers_['binary'].statistics_]})
print(fills)
print('Missing after preprocessing:', {name: int(frame.isna().sum().sum())
      for name, frame in [('train', X_train), ('validation', X_val), ('test', X_test)]})

### Duplicates

This part covers repeated records.

In [ ]:
duplicate_summary = {
    'repeated_ids_train': int(train_df['id'].duplicated().sum()),
    'repeated_records_excluding_id': int(train_df.drop(columns='id').duplicated().sum()),
    'repeated_feature_rows': int(raw_train.duplicated().sum()),
    'conflicting_feature_groups': int(train_df.groupby(
        FEATURE_COLUMNS, dropna=False)['is_canceled'].nunique().gt(1).sum()),
    'shared_raw_groups_train_val': len(set(groups.iloc[train_idx]) & set(groups.iloc[val_idx])),
}
print(duplicate_summary)
print('Repeated feature rows are retained as separate bookings.')

### Outliers

This part covers values far from the rest of the data.

In [ ]:
numeric = X_train_raw[NUMERIC_COLUMNS]
q1, q3 = numeric.quantile(.25), numeric.quantile(.75)
iqr = q3 - q1
outlier_review = pd.DataFrame({'median': numeric.median(), 'p99': numeric.quantile(.99),
                              'max': numeric.max(), 'skew': numeric.skew(), 'iqr': iqr,
                              'zero_pct': numeric.eq(0).sum() / numeric.notna().sum() * 100,
                              'outside_iqr': ((numeric < q1 - 1.5 * iqr) |
                                              (numeric > q3 + 1.5 * iqr)).sum()})
print(outlier_review.round(3))
plot_columns = ['days_before_arrival', 'nightly_rate_idr', 'prior_cancellations',
                'prior_completed_bookings', 'num_children', 'special_requests']
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for column, ax in zip(plot_columns, axes.flat):
    ax.hist(numeric[column].dropna(), bins=35, color='#28779e', edgecolor='white')
    ax.set_title(column)
    ax.set_ylabel('Bookings')
fig.suptitle('Training split before imputation')
fig.tight_layout()
plt.show()

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
audit_counts = {}
for name, frame in [('training', X_train_raw), ('validation', X_val_raw), ('test', raw_test)]:
    audit_counts[name] = {
        'zero_rate': int(frame['nightly_rate_idr'].eq(0).sum()),
        'zero_total_nights': int((frame['weekend_nights'] + frame['weekday_nights']).eq(0).sum()),
        'known_zero_total_guests': int((frame['num_adults'] + frame['num_children']).eq(0).sum()),
    }
print(pd.DataFrame(audit_counts).T)
print('Zero rates and zero-night stays are retained.')

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

<!-- rubric v3: derived columns, rate_per_guest_night -->

This part covers new attributes derived from existing ones.

In [ ]:
assert X_train.shape[1] == len(FEATURE_COLUMNS) == 17
print('No derived features added yet.')

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
print(X_train[NUMERIC_COLUMNS + BINARY_COLUMNS].agg(['min', 'max']).T)

### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
category_tables = []
for column in CATEGORICAL_COLUMNS:
    table = X_train.assign(is_canceled=y_train).groupby(column)['is_canceled'].agg(
        count='size', canceled_rate='mean').reset_index().rename(columns={column: 'category'})
    table.insert(0, 'feature', column)
    category_tables.append(table)
category_review = pd.concat(category_tables, ignore_index=True)
print(category_review.round(4))
unseen_categories = {column: {
    name: sorted(set(frame[column]) - set(X_train[column]))
    for name, frame in [('validation', X_val), ('test', X_test)]}
    for column in CATEGORICAL_COLUMNS}
print('Categories unseen during fitting:', unseen_categories)

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [ ]:
corr = X_train[NUMERIC_COLUMNS + BINARY_COLUMNS].corr()
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
correlation_review = pairs.rename('pearson_r').reset_index()
correlation_review.columns = ['feature_1', 'feature_2', 'pearson_r']
correlation_review = correlation_review.sort_values('pearson_r', key=lambda s: s.abs(), ascending=False)
print(correlation_review.head(10).round(4))
numeric_by_class = X_train.assign(is_canceled=y_train).groupby('is_canceled')[NUMERIC_COLUMNS].median().T
print(numeric_by_class)

## Class Balance

This part covers the proportion of each class in the training data.

In [ ]:
majority = int(y_train.mode().iloc[0])
baseline_pred = np.full(len(y_val), majority)
baseline = {'majority_class': majority,
            'validation_macro_f1': f1_score(y_val, baseline_pred, average='macro', labels=[0, 1], zero_division=0),
            'validation_accuracy': accuracy_score(y_val, baseline_pred)}
print(split_summary.round(3))
print(baseline)

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
for clean, raw in [(X_train, X_train_raw), (X_val, X_val_raw), (X_test, raw_test)]:
    assert clean.shape == raw.shape
    assert clean.columns.tolist() == FEATURE_COLUMNS
    assert clean.index.equals(raw.index)
    assert not clean.isna().any().any()
    assert np.isfinite(clean[NUMERIC_COLUMNS + BINARY_COLUMNS].to_numpy()).all()
    for column in NUMERIC_COLUMNS + BINARY_COLUMNS:
        known = raw[column].notna()
        np.testing.assert_array_equal(clean.loc[known, column], raw.loc[known, column])
assert set(X_train.columns).isdisjoint({'id', 'is_canceled'})
print('PASS: schema, row order, observed values, and missing values')

---

# Preparation Pipeline <a name="6"></a>

---

<!-- rubric v3: fitted steps, fitted_prep_steps -->

The pipeline combines the preparation steps so they run in one consistent order.

In [ ]:
from brownbear.preprocessing import make_numeric_preprocessor, make_discrete_preprocessor

numeric_prep = make_numeric_preprocessor().fit(train_set)
nb_prep = make_discrete_preprocessor(n_bins=5).fit(train_set)
X_train_numeric, X_val_numeric = numeric_prep.transform(train_set), numeric_prep.transform(val_set)
X_train_nb, X_val_nb = nb_prep.transform(train_set), nb_prep.transform(val_set)
assert X_train_numeric.shape[0] == X_train_nb.shape[0] == len(train_set)
assert X_val_numeric.shape[0] == X_val_nb.shape[0] == len(val_set)
assert np.isfinite(X_val_numeric).all() and (X_val_nb >= 0).all()
print('Basic:', X_train.shape, X_val.shape, X_test.shape)
print('Numeric:', X_train_numeric.shape, X_val_numeric.shape)
print('Categorical NB:', X_train_nb.shape, X_val_nb.shape)

---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

<!-- rubric v3: pairwise distances, val_neighbor_dists -->

This part covers the nearest neighbor classifier and its validation.

In [ ]:
# Write your code here

## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
# Write your code here

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [ ]:
# Write your code here

## Support Vector Machine

<!-- rubric v3: total slack, slack_total -->

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
# Write your code here

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
# Write your code here

## Model Evaluation

<!-- figure palette: #6d2fa3 -->

This part compares the validation results of all models.

In [ ]:
# Write your code here

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
# Write your code here

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

In [ ]:
# Write your code here

---

# Saving and Loading Models <a name="9"></a>

---

<!-- artifacts: <model>_v3.pkl -->

This section stores the trained models and restores them for prediction.

In [ ]:
# Write your code here

---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
# Write your code here

---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.

## References

- [StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)
- [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)
- [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)
- [KBinsDiscretizer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html)
- [Categorical Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html#categorical-naive-bayes)
